# Binance Intra Arb01 - Persist Data Pull

Read all columns for all supported persist tables from `binance-intra-arb01` through the local persist read server. The server is configured with `max_window_sec = 3600`, so the notebook pulls the selected range in 1 hour chunks and concatenates each table into a DataFrame.

Default range is the latest 8 hours. Set `START` and `END` in the setup cell to pull a specific wider range.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts' / 'persist_read_client.py').exists():
    REPO_ROOT = Path('/home/ubuntu/crypto_mkt/mkt_signal')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.persist_read_client import PersistReadClient, iter_windows

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

BASE_URL = 'http://127.0.0.1:8822'
SOURCE_ID = 'binance-intra-arb01'
TABLES = (
    'uniform_orders',
    'order_updates_unmatched',
    'trade_updates_unmatched',
)

# Default pull range. Leave START/END as None for latest WINDOW_HOURS.
WINDOW_HOURS = 8
START = None  # Example: datetime(2026, 6, 11, 0, 0, tzinfo=timezone.utc)
END = None

CHUNK_SECONDS = 3600
REQUEST_TIMEOUT_SEC = 120

client = PersistReadClient(BASE_URL, timeout_sec=REQUEST_TIMEOUT_SEC, tz='UTC')

## Health And Schema

In [ ]:
print(client.health())

schemas = {}
for table in TABLES:
    schema = client.schema(table, source_id=SOURCE_ID)
    schemas[table] = schema
    print(f'{table}: source_id={SOURCE_ID} columns={len(schema.columns)} formats={schema.formats}')
    print(list(schema.columns))

## Resolve Pull Window

In [ ]:
def as_utc(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = datetime.fromisoformat(value.replace('Z', '+00:00'))
    if value.tzinfo is None:
        return value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc)


end = as_utc(END) or datetime.now(timezone.utc)
start = as_utc(START) or (end - timedelta(hours=WINDOW_HOURS))
if end <= start:
    raise ValueError(f'END must be greater than START: {start} -> {end}')

print('UTC window:', start.isoformat(), '->', end.isoformat())
print('chunk_seconds:', CHUNK_SECONDS)

## Pull All Tables

In [ ]:
def read_table_range(table, start, end):
    frames = []
    chunk_counts = []
    for window in iter_windows(start, end, window_sec=CHUNK_SECONDS, tz=timezone.utc):
        chunk_start = datetime.fromtimestamp(window.start_us / 1_000_000, timezone.utc)
        chunk_end = datetime.fromtimestamp(window.end_us / 1_000_000, timezone.utc)
        df = client.read_pandas(
            table,
            start_us=window.start_us,
            end_us=window.end_us,
            source_id=SOURCE_ID,
            columns=None,
            timeout_sec=REQUEST_TIMEOUT_SEC,
        )
        frames.append(df)
        chunk_counts.append({
            'table': table,
            'start_utc': chunk_start,
            'end_utc': chunk_end,
            'rows': len(df),
        })
        print(f'{table} {chunk_start.isoformat()} -> {chunk_end.isoformat()} rows={len(df):,}')

    data = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if 'ts_us' in data.columns:
        data['ts'] = pd.to_datetime(data['ts_us'], unit='us', utc=True)
        data = data.sort_values('ts_us').reset_index(drop=True)
    if not data.empty:
        data.insert(0, 'source_id', SOURCE_ID)
        data.insert(1, 'table', table)
    return data, pd.DataFrame(chunk_counts)


datasets = {}
chunk_reports = []
for table in TABLES:
    data, report = read_table_range(table, start, end)
    datasets[table] = data
    chunk_reports.append(report)

binance_uniform_orders = datasets['uniform_orders']
binance_order_updates_unmatched = datasets['order_updates_unmatched']
binance_trade_updates_unmatched = datasets['trade_updates_unmatched']
read_report = pd.concat(chunk_reports, ignore_index=True) if chunk_reports else pd.DataFrame()

summary = pd.DataFrame([
    {'table': table, 'rows': len(df), 'columns': len(df.columns)}
    for table, df in datasets.items()
])
summary

In [ ]:
read_report

## Quick Checks

In [ ]:
def top_groups(df, columns, n=30):
    columns = [c for c in columns if c in df.columns]
    if df.empty or not columns:
        return pd.DataFrame(columns=columns + ['rows'])
    return (
        df.groupby(columns, dropna=False)
        .size()
        .reset_index(name='rows')
        .sort_values('rows', ascending=False)
        .head(n)
    )


top_groups(binance_uniform_orders, ['symbol', 'venue', 'status'])

In [ ]:
top_groups(binance_order_updates_unmatched, ['symbol', 'venue', 'status'])

In [ ]:
top_groups(binance_trade_updates_unmatched, ['symbol', 'venue', 'status'])

## Full DataFrames

These variables contain the full selected range for Binance intra arb01:

- `binance_uniform_orders`
- `binance_order_updates_unmatched`
- `binance_trade_updates_unmatched`

In [ ]:
binance_uniform_orders

In [ ]:
binance_order_updates_unmatched

In [ ]:
binance_trade_updates_unmatched

## Optional Export

In [ ]:
# Set to True when you want local parquet snapshots under order_exports/.
EXPORT_PARQUET = False

if EXPORT_PARQUET:
    export_dir = REPO_ROOT / 'order_exports' / SOURCE_ID / f'{start:%Y%m%dT%H%M%SZ}_{end:%Y%m%dT%H%M%SZ}'
    export_dir.mkdir(parents=True, exist_ok=True)
    for table, df in datasets.items():
        export_path = export_dir / f'{table}.parquet'
        df.to_parquet(export_path, index=False)
        print(table, len(df), export_path)
else:
    print('Set EXPORT_PARQUET = True to write parquet snapshots.')